## Setup

Import the libraries used throughout the notebook and initialize the OpenAI client.


In [ ]:
import pandas as pd
import json
import re
import random
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

client = OpenAI(api_key='')


## Load Data

Read the manually labeled conversation CSV into a DataFrame so the evaluation pipeline can use the ground-truth labels.


In [ ]:
# Read in the CSV file
df = pd.read_csv('/Users/sanyakhan/GE-synthetic-chatbot/dia12_manual.csv')
df

## Run LLM Evaluation

Group each conversation, ask the model to predict labels and explanations, compare predictions against the labeled data, and save the model evaluation results.


In [ ]:


# topic switch helper

def compute_true_switch(subcats):
    for i in range(1, len(subcats)):
        if subcats[i] != subcats[i - 1]:
            return i + 1  # 1-based indexing
    return None


# group conversations

convos = []

for convo_id, group in df.groupby("conversation_id"):
    group = group.sort_values("turn")

    user_turns = group["user_message"].dropna().tolist()
    subcats_per_turn = group["subcategory"].tolist()

    row = group.iloc[0]

    convos.append({
        "conversation_id": convo_id,
        "user_turns": user_turns,
        "subcats_per_turn": subcats_per_turn,
        "tone": row["tone"],
        "barrier": row["barrier"],
        "age": row.get("age", None),
        "topic_shift_turn": compute_true_switch(subcats_per_turn),
        "freeform": None if pd.isna(row.get("free_form", None)) else row.get("free_form", None),
        "edge_case": None if pd.isna(row.get("edge_case", None)) else row.get("edge_case", None),
    })


# safe json

def safe_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\[.*\]", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        else:
            raise ValueError(f"No JSON found:\n{text}")


# evaluate batch

def evaluate_batch(batch):

    formatted = []

    for i, convo in enumerate(batch):

        turns = "\n".join([
            f"TURN {idx + 1}: {t}" for idx, t in enumerate(convo["user_turns"])
        ])

        formatted.append(f"""
ID: {i}

EDGE CASE:
{convo['edge_case']}

FREEFORM CONTEXT:
{convo['freeform']}

USER TURNS:
{turns}
""")

    prompt = f"""
Evaluate each conversation.

Return a LIST of JSON objects in order.

OUTPUT FORMAT:
[
{{
    "id": int,

    "predicted_subcategories_per_turn": [],
    "subcategory_explanation": "",

    "predicted_tone": "",
    "tone_explanation": "",

    "predicted_barrier": "",
    "barrier_explanation": "",

    "predicted_age": "teen/young_adult/adult/unknown",
    "age_explanation": "",

    "predicted_topic_switch_turn": int or null,
    "topic_switch_explanation": "",

    "freeform_detected": true/false/null,
    "freeform_explanation": "",

    "edge_case_detected": true/false/null,
    "edge_case_explanation": ""
}}
]

========================
LABEL OPTIONS (STRICT)
========================

SUBCATEGORY:
puberty_and_body_changes
menstruation_and_cycle_issues
pregnancy
abortion
contraception
stis
hiv_aids
intimate_relationships_and_consent
srh_services

TONE DEFINITIONS:
- none_neutral_just_curious: calm, informational, no strong emotion
- fear_shame_stigma: embarrassed, afraid of judgment, stigma-aware
- anxiety_overwhelm: stressed, urgent, overloaded
- positive_health_seeking: proactive, responsible, solution-oriented
- confused: explicitly unsure, asks for clarification

BARRIER DEFINITIONS:
- knowledge_barrier: lacks understanding or information
- practical_barrier: cost, access, logistics issues
- social_norms_external_barrier: stigma, culture, fear of others
- social_support_barrier: lacks support or guidance
- internal_struggle_barrier: personal hesitation or internal conflict


========================
RULES
========================

- Use ONLY USER TURNS
- Output EXACT label strings
- Do NOT include numbers
- Be strict

========================
SUBCATEGORY
========================

- Assign one subcategory label to each user turn
- Use ONLY the label options
- Do not calculate a match scorerect

========================
TOPIC SWITCH
========================

- Turn numbering starts at 1 (first user message = Turn 1)
- Find the FIRST TURN where the subcategory changes
- Return that turn number

- Do not calculate topic_switch_match
- Only predict the first turn where the topic changes

========================
AGE
========================

Goal:
Infer the user's age group based ONLY on user turns.

Output:
- "teen"
- "young_adult"
- "adult"
- "unknown"

Rules:
- Do NOT default to "unknown" unless there is truly no signal
- You MUST attempt inference when any contextual clue exists

Inference Heuristics:

TEEN:
- Mentions parents controlling decisions
- Mentions school (not college)
- Asks basic puberty/body questions
- Tone suggests dependency or lack of autonomy

YOUNG_ADULT:
- Mentions college, dating, early independence
- Questions about contraception, relationships, identity
- No mention of children

ADULT:
- Mentions having children
- Mentions marriage, long-term partner, or healthcare decisions
- References work-life responsibilities

UNKNOWN:
- Only use if NO contextual clues exist

Important:
- Base inference ONLY on user messages
- Do NOT use assistant responses
- Do NOT overthink, use best reasonable guess

========================
FREEFORM
========================

Goal:
Determine whether the USER'S message reflects the CONSTRAINT described in FREEFORM.

FREEFORM is open-ended and may describe:
- financial constraints
- access issues
- social/family constraints
- emotional or personal limitations

You must interpret the MEANING, not exact wording.

TRUE if:
- the user's behavior, concerns, or constraints align with the situation described in FREEFORM
- even if phrased differently

FALSE if:
- no meaningful connection

Important:
- Do NOT rely on keyword matching
- Focus on underlying meaning
- Prefer TRUE when there is a reasonable semantic match

========================
EXPLANATIONS
========================

- Must reference turns
- Must justify decisions

========================
CONVERSATIONS:
{''.join(formatted)}
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
    )

    return safe_json(response.choices[0].message.content)


# batching

def batchify(lst, batch_size=5):
    return [lst[i:i + batch_size] for i in range(0, len(lst), batch_size)]


# normalize

def normalize_none(pred, gt):
    if pd.isna(gt) or gt in [None, "none", "None"]:
        return None
    return pred


def hybrid_match(pred, gt):
    # case 1: both missing -> True
    if gt is None:
        return pred is None

    # case 2: gt exists -> compare presence
    return pred == True


def compute_subcat_match(pred, gt):
    if pred is None or gt is None:
        return 0

    if len(pred) != len(gt):
        return 0

    return sum(p == g for p, g in zip(pred, gt)) / len(gt)


def compute_exact_match(pred, gt):
    if pd.isna(gt):
        gt = None
    if pd.isna(pred):
        pred = None
    return pred == gt


# parallel execution

results = []
batches = batchify(convos, batch_size=5)

with ThreadPoolExecutor(max_workers=5) as executor:

    future_to_batch = {
        executor.submit(evaluate_batch, batch): batch
        for batch in batches
    }

    for future in as_completed(future_to_batch):

        batch = future_to_batch[future]
        batch_results = future.result()

        for res in batch_results:
            convo = batch[res["id"]]

            results.append({
                "conversation_id": convo["conversation_id"],

                # predictions
                "pred_subcats": res["predicted_subcategories_per_turn"],
                "pred_tone": res["predicted_tone"],
                "pred_barrier": res["predicted_barrier"],
                "pred_age": res["predicted_age"],
                "pred_topic_switch": res["predicted_topic_switch_turn"],

                # matches
                "tone_match": compute_exact_match(
                    res["predicted_tone"],
                    convo["tone"]
                ),

                "barrier_match": compute_exact_match(
                    res["predicted_barrier"],
                    convo["barrier"]
                ),

                "topic_switch_match": compute_exact_match(
                    res["predicted_topic_switch_turn"],
                    convo["topic_shift_turn"]
                ),

                "freeform_match": hybrid_match(
                    normalize_none(res["freeform_detected"], convo["freeform"]),
                    convo["freeform"]
                ),

                "edge_case_match": hybrid_match(
                    normalize_none(res["edge_case_detected"], convo["edge_case"]),
                    convo["edge_case"]
                ),

                # explanations
                "subcategory_expl": res["subcategory_explanation"],
                "tone_expl": res["tone_explanation"],
                "barrier_expl": res["barrier_explanation"],
                "age_expl": res["age_explanation"],
                "topic_switch_expl": res["topic_switch_explanation"],
                "freeform_expl": res["freeform_explanation"],
                "edge_case_expl": res["edge_case_explanation"],
                "user_turns": convo["user_turns"]
            })

results_df = pd.DataFrame(results)


# save

results_df.to_csv("eval_results13.csv", index=False)

print(results_df.head())


## Measure Theme Diversity

Extract normalized concern themes from each conversation, score diversity within and across conversations, cache theme outputs, and save the diversity results.


In [ ]:
# group conversations

convos = []

for convo_id, group in df.groupby("conversation_id"):
    user_turns = group["user_message"].dropna().tolist()

    convos.append({
        "conversation_id": convo_id,
        "user_turns": user_turns,
        "subcategory": group.iloc[0]["subcategory"],
        "barrier": group.iloc[0]["barrier"],
        "tone": group.iloc[0]["tone"]
    })


# step 1: llm themes extract and normalize

def extract_themes(user_turns):

    prompt = f"""
    Analyze this user conversation.

    Identify the DIFFERENT TYPES OF CONCERNS the user expresses.

    Normalize them into GENERAL categories so they are consistent across conversations.

    Example categories:
    - financial
    - legal
    - social stigma
    - access
    - medical safety
    - emotional concern
    - misinformation

    RULES:
    - Return 2-5 themes
    - Keep them SHORT (1-3 words)
    - Use CONSISTENT wording across conversations
    - No duplicates
    - Base ONLY on USER turns

    USER TURNS:
    {user_turns}

    Return JSON:
    {{
        "themes": []
    }}
    """

    res = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    try:
        return json.loads(res.output_text)["themes"]
    except:
        return []


# step 2: within-convo diversity

def within_diversity_score(themes):
    if len(themes) == 0:
        return 0
    return len(set(themes)) / 5


def explain_within(themes):
    unique = list(set(themes))

    if len(unique) == 0:
        return "No clear themes detected"
    elif len(unique) == 1:
        return f"Low diversity: focuses only on {unique[0]}"
    elif len(unique) <= 3:
        return f"Moderate diversity: covers {', '.join(unique)}"
    else:
        return f"High diversity: spans multiple concern types ({', '.join(unique)})"


# step 3: between-convo diversity

def between_diversity(convos):

    all_themes = []

    for c in convos:
        all_themes.extend(c["themes"])

    if len(all_themes) == 0:
        return 0

    counts = Counter(all_themes)
    total = sum(counts.values())

    probs = [v / total for v in counts.values()]

    entropy = -sum(p * math.log(p) for p in probs if p > 0)
    max_entropy = math.log(len(counts)) if len(counts) > 1 else 1

    return entropy / max_entropy


def explain_between(score):

    if score < 0.3:
        return "Low diversity: conversations are very similar"
    elif score < 0.7:
        return "Moderate diversity: some variation across conversations"
    else:
        return "High diversity: wide range of concerns covered"


# cache

CACHE_FILE = "theme_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)
else:
    cache = {}


# run pipeline

for i, c in enumerate(convos):
    cid = c["conversation_id"]

    print(f"{i + 1}/{len(convos)}")

    if cid in cache:
        themes = cache[cid]
    else:
        themes = extract_themes(c["user_turns"])
        cache[cid] = themes

        with open(CACHE_FILE, "w") as f:
            json.dump(cache, f)

    c["themes"] = themes
    c["within_score"] = within_diversity_score(themes)
    c["within_explanation"] = explain_within(themes)


# between

between_score = between_diversity(convos)
between_explanation = explain_between(between_score)


# output

rows = []

for c in convos:
    rows.append({
        "conversation_id": c["conversation_id"],
        "subcategory": c["subcategory"],
        "barrier": c["barrier"],
        "tone": c["tone"],
        "themes": ", ".join(c["themes"]),
        "within_diversity_score": c["within_score"],
        "within_explanation": c["within_explanation"],
        "between_diversity_score": between_score,
        "between_explanation": between_explanation
    })

diversity_df = pd.DataFrame(rows)

diversity_df.to_csv("diversity_results_13.csv", index=False)

print("DONE")
print("Between diversity score:", between_score)
print("Explanation:", between_explanation)

1/10
2/10
3/10
4/10
5/10
6/10
7/10
8/10
9/10
10/10
DONE
Between diversity score: 0.9456798789256272
Explanation: High diversity: wide range of concerns covered


## Check Turn-Level Subcategory Accuracy

Explode predicted subcategories to one row per turn, align them with the labeled turns, and calculate the turn-level match rate.


In [ ]:
# subcategory matches

pred_df = results_df[["conversation_id", "pred_subcats"]].explode("pred_subcats")
pred_df = pred_df.rename(columns={"pred_subcats": "pred_subcategory"})

# add turn ids

df["turn_id"] = df.groupby("conversation_id").cumcount()
pred_df["turn_id"] = pred_df.groupby("conversation_id").cumcount()

# merge gt and pred

merged = df.merge(
    pred_df,
    on=["conversation_id", "turn_id"],
    how="left"
)

# match

merged["subcategory_match"] = (
    merged["subcategory"] == merged["pred_subcategory"]
)

# metrics

accuracy = merged["subcategory_match"].mean()
num_matches = merged["subcategory_match"].sum()
total = len(merged)

print(f"Accuracy: {accuracy}")
print(f"{num_matches} / {total} matched")

# optional: per convo

per_convo = merged.groupby("conversation_id")["subcategory_match"].mean()

# sanity check

print(merged[[
    "conversation_id",
    "turn_id",
    "subcategory",
    "pred_subcategory",
    "subcategory_match"
]].head(20)) 


Accuracy: 0.38571428571428573
27 / 70 matched
   conversation_id  turn_id               subcategory  \
0           conv_1        0  puberty_and_body_changes   
1           conv_1        1  puberty_and_body_changes   
2           conv_1        2  puberty_and_body_changes   
3           conv_1        3  puberty_and_body_changes   
4           conv_1        4  puberty_and_body_changes   
..             ...      ...                       ...   
15          conv_3        1  puberty_and_body_changes   
16          conv_3        2  puberty_and_body_changes   
17          conv_3        3  puberty_and_body_changes   
18          conv_3        4  puberty_and_body_changes   
19          conv_3        5  puberty_and_body_changes   

                 pred_subcategory  subcategory_match  
0        puberty_and_body_changes               True  
1        puberty_and_body_changes               True  
2                    srh_services              False  
3        puberty_and_body_changes               T

## Summarize Conversation-Level Matches

Print match rates for conversation-level labels such as tone, barrier, topic switch, and freeform detection.


In [135]:
match_cols = [
    "tone_match",
    "barrier_match",
    "topic_switch_match",
    "freeform_match"
]

total = len(results_df)

for col in match_cols:
    ratio = results_df[col].sum() / total
    print(col, ":", ratio)

tone_match : 1.0
barrier_match : 0.0
topic_switch_match : 0.2
freeform_match : 1.0


## Compare Age Predictions

Build a small table comparing the labeled age value with the model-predicted age group for each conversation.


In [136]:
gt_age = (
    df.groupby("conversation_id")["age"]
    .first()
    .reset_index()
    .rename(columns={"age": "true_age"})
)
pred_age = results_df[["conversation_id", "pred_age"]]
age_compare = gt_age.merge(pred_age, on="conversation_id")
age_compare.head(10)

,conversation_id,true_age,pred_age
0,conv_1,14,teen
1,conv_10,14,teen
2,conv_2,14,teen
3,conv_3,14,teen
4,conv_4,14,teen
5,conv_5,14,teen
6,conv_6,14,teen
7,conv_7,14,teen
8,conv_8,14,teen
9,conv_9,14,teen


## Create Final Metrics Table

Combine subcategory accuracy, conversation-level matches, age comparison, and diversity scores into one final evaluation summary.


In [ ]:
# turn-level subcategory

pred_df = results_df[["conversation_id", "pred_subcats"]].explode("pred_subcats")
pred_df = pred_df.rename(columns={"pred_subcats": "pred_subcategory"})

# add turn ids

df["turn_id"] = df.groupby("conversation_id").cumcount()
pred_df["turn_id"] = pred_df.groupby("conversation_id").cumcount()

# merge ground truth and predictions

merged = df.merge(
    pred_df,
    on=["conversation_id", "turn_id"],
    how="left"
)

# match

merged["subcategory_match"] = (
    merged["subcategory"] == merged["pred_subcategory"]
)

# overall accuracy

accuracy = merged["subcategory_match"].mean()
print("Subcategory accuracy:", accuracy)

# per-convo subcategory accuracy

per_convo = (
    merged.groupby("conversation_id")["subcategory_match"]
    .mean()
    .reset_index()
    .rename(columns={"subcategory_match": "subcat_accuracy"})
)


# convo-level metrics

convo_metrics = results_df[[
    "conversation_id",
    "tone_match",
    "barrier_match",
    "topic_switch_match",
    "freeform_match",
    "edge_case_match",
    "pred_age"
]]


# true age

true_age = (
    df.groupby("conversation_id")["age"]
    .first()
    .reset_index()
    .rename(columns={"age": "true_age"})
)


# final merge

final_df = convo_metrics.merge(true_age, on="conversation_id")
final_df = final_df.merge(per_convo, on="conversation_id")

# age match

final_df["age_match"] = final_df["true_age"] == final_df["pred_age"]


# add diversity

final_df = final_df.merge(
    diversity_df[[
        "conversation_id",
        "within_diversity_score",
        "between_diversity_score"
    ]],
    on="conversation_id",
    how="left"
)


# metrics summary

for col in [
    "tone_match",
    "barrier_match",
    "topic_switch_match",
    "freeform_match",
    "edge_case_match",
    "age_match"
]:
    print(col, ":", final_df[col].mean())

# overall dataset diversity

print("overall_diversity_score :", final_df["between_diversity_score"].iloc[0])


# inspect

print(final_df[[
    "conversation_id",
    "true_age",
    "pred_age",
    "subcat_accuracy",
    "within_diversity_score"
]].head(10))


Subcategory accuracy: 0.38571428571428573
tone_match : 1.0
barrier_match : 0.0
topic_switch_match : 0.2
freeform_match : 1.0
edge_case_match : 1.0
age_match : 0.0
overall_diversity_score : 0.9456798789256272
  conversation_id  true_age pred_age  subcat_accuracy  within_diversity_score
0          conv_1        14     teen         0.428571                     0.8
1         conv_10        14     teen         1.000000                     1.0
2          conv_2        14     teen         0.000000                     0.8
3          conv_3        14     teen         0.000000                     0.8
4          conv_4        14     teen         0.142857                     0.8
5          conv_5        14     teen         0.142857                     1.0
6          conv_6        14     teen         0.714286                     0.8
7          conv_7        14     teen         0.857143                     0.8
8          conv_8        14     teen         0.000000                     1.0
9          c